# HTF Trend Alignment Screen (NQ, training slice)

Pre-registration: `research/hypotheses/htf_alignment_edge.md` (locked in commit `f848afb`).

**Rules of this notebook:**
- Training slice only (2018-01-02 → 2022-12-30). Test slice sealed.
- Co-primary horizons: 15m AND 1h (Bonferroni-adjusted 97.5% CIs).
- No parameter changes. MA lengths (1H MA(20), 4H MA(16)), observation window (09:30–15:00 ET), HAC lag (23 bars), significance bar (97.5%), and pass criteria are all fixed by the pre-registration.
- Pass/fail is evaluated mechanically at the end. No negotiation.


In [ ]:
import sys
from pathlib import Path
from datetime import time as dt_time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make engine/ importable from notebook location
repo_root = Path.cwd().resolve().parents[1]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from engine.data_loader import load_csv, split_train_test

# Locked constants from pre-registration
MA_1H_LEN = 20
MA_4H_LEN = 16
HAC_LAG = 23
Z_97_5 = 2.241402727604947  # scipy.stats.norm.ppf(0.9875) — two-sided 97.5% CI
OBS_WINDOW_START = dt_time(9, 30)
OBS_WINDOW_END = dt_time(15, 0)
HALF_CUTOFF = pd.Timestamp("2020-07-01", tz="America/New_York")

pd.set_option("display.float_format", lambda x: f"{x:+.5%}" if abs(x) < 1 else f"{x:.4f}")


## 1. Load and split
Use `engine.data_loader.load_csv` + `split_train_test`. Drop the test DataFrame from memory immediately so there's no accidental exposure.


In [ ]:
df_all = load_csv("../../data/nq_15m_data.csv")
df_train, df_test = split_train_test(df_all)
del df_all, df_test  # explicit: test slice not touched in this screen

print(f"Training slice: {df_train.index.min()} -> {df_train.index.max()}")
print(f"Training bars:  {len(df_train):,}")
print(f"RTH bars:       {df_train['is_rth'].sum():,}")


## 2. Build HTF state (1H + 4H)

Steps:
1. Resample 15-min bars to 1H and 4H (OHLCV aggregation).
2. Compute `MA(20)` on 1H close, `MA(16)` on 4H close.
3. Compute simple slopes: `MA_t − MA_{t−1}` on each timeframe.
4. Sign the slopes into `{+1, 0, −1}` states.
5. **Shift each state forward by one of its own bars** (`.shift(1)`) — critical lookahead protection.
6. Reindex onto the 15-min timeline with forward-fill.

The shift means: at any 15-min timestamp, the HTF state reflects only the **most recently closed** 1H and 4H bar, never the one currently forming.


In [ ]:
AGG = {"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}

# --- 1H ---
htf_1h = df_train.resample("1h", label="left", closed="left").agg(AGG).dropna(subset=["close"])
htf_1h["ma"] = htf_1h["close"].rolling(MA_1H_LEN).mean()
htf_1h["slope"] = htf_1h["ma"].diff()
htf_1h["state_raw"] = np.sign(htf_1h["slope"]).fillna(0).astype(int)
# Shift by one 1H bar to ensure lookahead safety
htf_1h["state"] = htf_1h["state_raw"].shift(1)

# --- 4H ---
htf_4h = df_train.resample("4h", label="left", closed="left").agg(AGG).dropna(subset=["close"])
htf_4h["ma"] = htf_4h["close"].rolling(MA_4H_LEN).mean()
htf_4h["slope"] = htf_4h["ma"].diff()
htf_4h["state_raw"] = np.sign(htf_4h["slope"]).fillna(0).astype(int)
htf_4h["state"] = htf_4h["state_raw"].shift(1)

print(f"1H bars: {len(htf_1h):,}, non-NaN state: {htf_1h['state'].notna().sum():,}")
print(f"4H bars: {len(htf_4h):,}, non-NaN state: {htf_4h['state'].notna().sum():,}")


In [ ]:
# Align to 15-min timeline via forward-fill
state_1h_15m = htf_1h["state"].reindex(df_train.index, method="ffill")
state_4h_15m = htf_4h["state"].reindex(df_train.index, method="ffill")

df = df_train.copy()
df["state_1h"] = state_1h_15m
df["state_4h"] = state_4h_15m

# Composite HTF state
def composite(s1, s4):
    if pd.isna(s1) or pd.isna(s4):
        return np.nan
    if s1 == 1 and s4 == 1:
        return 1
    if s1 == -1 and s4 == -1:
        return -1
    return 0

df["htf_state"] = [composite(a, b) for a, b in zip(df["state_1h"], df["state_4h"])]

print("Composite state distribution (all 15-min bars):")
print(df["htf_state"].value_counts(dropna=False).sort_index())


## 3. Forward returns and observation window

Three forward-return horizons (simple returns on close):
- **15m** (1 bar) — co-primary
- **1h** (4 bars) — co-primary
- **4h** (16 bars) — diagnostic only

Observation window: bar open time in `[09:30, 15:00]` ET, so the 1h forward horizon terminates at or before 16:00 RTH close for every observation bar.


In [ ]:
df["fwd_15m"] = df["close"].shift(-1) / df["close"] - 1
df["fwd_1h"] = df["close"].shift(-4) / df["close"] - 1
df["fwd_4h"] = df["close"].shift(-16) / df["close"] - 1

# Observation window: 09:30 <= bar_time <= 15:00
bar_times = df.index.time
obs_mask = (bar_times >= OBS_WINDOW_START) & (bar_times <= OBS_WINDOW_END)
df["in_obs"] = obs_mask

# Composite state must be non-NaN AND forward horizons must be non-NaN AND in obs window
obs = df[df["in_obs"] & df["htf_state"].notna() & df["fwd_15m"].notna() & df["fwd_1h"].notna()].copy()
obs["htf_state"] = obs["htf_state"].astype(int)
obs["state_1h"] = obs["state_1h"].astype(int)
obs["state_4h"] = obs["state_4h"].astype(int)

print(f"Observation bars after mask + non-NaN features: {len(obs):,}")
print(f"Date range: {obs.index.min()} -> {obs.index.max()}")
print()
print("Per-bucket counts:")
print(obs["htf_state"].value_counts().sort_index())


## 4. De-mean against unconditional mean

For each horizon, subtract the unconditional mean forward return across the observation window. This strips NQ's ~2–3 bps unconditional intraday drift (2018–2022) and leaves the **conditional** effect.


In [ ]:
uncond = {
    "fwd_15m": obs["fwd_15m"].mean(),
    "fwd_1h":  obs["fwd_1h"].mean(),
}
if obs["fwd_4h"].notna().any():
    uncond["fwd_4h"] = obs.loc[obs["fwd_4h"].notna(), "fwd_4h"].mean()

print("Unconditional means (pre-de-meaning):")
for k, v in uncond.items():
    print(f"  {k}: {v:+.5%}")

obs["demean_15m"] = obs["fwd_15m"] - uncond["fwd_15m"]
obs["demean_1h"]  = obs["fwd_1h"]  - uncond["fwd_1h"]
if "fwd_4h" in uncond:
    obs["demean_4h"] = obs["fwd_4h"] - uncond["fwd_4h"]


## 5. Newey-West HAC standard errors

For a sample mean estimator, the HAC long-run variance is

$$\sigma^2_{\text{LR}} = \gamma_0 + 2 \sum_{k=1}^{L} w_k \gamma_k,\quad w_k = 1 - \tfrac{k}{L+1}$$

where $\gamma_k$ is the lag-$k$ autocovariance of the series and $w_k$ is the Bartlett weight. The SE of the mean is $\sqrt{\sigma^2_{\text{LR}}/n}$.

Truncation lag `L = 23` (one trading day within the observation window), locked in the pre-reg.


In [ ]:
def hac_se(y: np.ndarray, lag: int = HAC_LAG) -> float:
    """Newey-West (Bartlett) HAC standard error of the sample mean."""
    y = np.asarray(y, dtype=float)
    n = len(y)
    if n <= lag + 1:
        return float("nan")
    u = y - y.mean()
    gamma0 = np.mean(u * u)
    lrvar = gamma0
    for k in range(1, lag + 1):
        w = 1.0 - k / (lag + 1)
        gamma_k = np.mean(u[k:] * u[:-k])
        lrvar += 2.0 * w * gamma_k
    if lrvar < 0:
        # Newey-West guarantees non-negative with Bartlett, but guard anyway
        lrvar = gamma0
    return float(np.sqrt(lrvar / n))


def bucket_stats(group: pd.DataFrame, col: str) -> dict:
    y = group[col].dropna().values
    if len(y) < HAC_LAG + 2:
        return {"n": len(y), "mean": float("nan"), "se": float("nan"),
                "ci_lo": float("nan"), "ci_hi": float("nan"), "excl_zero": False}
    m = y.mean()
    se = hac_se(y)
    lo, hi = m - Z_97_5 * se, m + Z_97_5 * se
    return {"n": len(y), "mean": m, "se": se, "ci_lo": lo, "ci_hi": hi,
            "excl_zero": (lo > 0) or (hi < 0)}


# Per-bucket, per-horizon stats (full training slice)
horizons = ["demean_15m", "demean_1h"]
rows = []
for state, g in obs.groupby("htf_state"):
    for h in horizons:
        s = bucket_stats(g, h)
        s["state"] = int(state)
        s["horizon"] = h
        rows.append(s)

stats_df = pd.DataFrame(rows).set_index(["state", "horizon"]).sort_index()
print("Per-bucket, per-co-primary-horizon stats (97.5% HAC CI):")
print(stats_df)


## 6. Diagnostic 2×2 interaction table (1h horizon)

Not used for pass criteria — reported for transparency. If the agreement buckets (`++`, `−−`) don't differ cleanly from the conflict buckets (`+−`, `−+`), the dual-filter is decorative.


In [ ]:
pivot = obs.pivot_table(
    index="state_1h", columns="state_4h", values="demean_1h", aggfunc="mean"
)
pivot = pivot.reindex(index=[1, 0, -1], columns=[1, 0, -1])
print("1h-horizon de-meaned mean return, by (state_1h x state_4h):")
print(pivot)

count_pivot = obs.pivot_table(
    index="state_1h", columns="state_4h", values="demean_1h", aggfunc="count"
).reindex(index=[1, 0, -1], columns=[1, 0, -1])
print()
print("Counts:")
print(count_pivot)


## 7. Stability — Half A vs Half B

- **Half A:** 2018-01-02 → 2020-06-30
- **Half B:** 2020-07-01 → 2022-12-30

For the pattern to be stable, the `+1` and `−1` bucket mean signs must be preserved in both halves on both co-primary horizons.


In [ ]:
half_a = obs[obs.index < HALF_CUTOFF]
half_b = obs[obs.index >= HALF_CUTOFF]
print(f"Half A: {len(half_a):,} bars, {half_a.index.min()} -> {half_a.index.max()}")
print(f"Half B: {len(half_b):,} bars, {half_b.index.min()} -> {half_b.index.max()}")

def half_means(half_df):
    out = {}
    for state, g in half_df.groupby("htf_state"):
        out[int(state)] = {
            "demean_15m": g["demean_15m"].mean(),
            "demean_1h": g["demean_1h"].mean(),
            "n": len(g),
        }
    return out

a_stats = half_means(half_a)
b_stats = half_means(half_b)

print()
print("Half A means:")
for s in sorted(a_stats.keys()):
    r = a_stats[s]
    print(f"  state={s:+d}: n={r['n']:>6,d} | 15m={r['demean_15m']:+.5%} | 1h={r['demean_1h']:+.5%}")
print("Half B means:")
for s in sorted(b_stats.keys()):
    r = b_stats[s]
    print(f"  state={s:+d}: n={r['n']:>6,d} | 15m={r['demean_15m']:+.5%} | 1h={r['demean_1h']:+.5%}")


## 8. Pass / fail evaluation (mechanical)

All four must hold:

1. `+1` bucket: de-meaned mean > 0 AND 97.5% HAC CI excludes zero, on **both** 15m and 1h.
2. `−1` bucket: de-meaned mean < 0 AND 97.5% HAC CI excludes zero, on **both** 15m and 1h.
3. `0` bucket 1h mean is strictly between the `+1` and `−1` bucket 1h means.
4. `+1` and `−1` bucket signs preserved in both Half A and Half B on both co-primary horizons.


In [ ]:
def get(s, h):
    return stats_df.loc[(s, h)]

p1_15 = get(1, "demean_15m")
p1_1h = get(1, "demean_1h")
m1_15 = get(-1, "demean_15m")
m1_1h = get(-1, "demean_1h")
z0_1h = get(0, "demean_1h")

crit1 = (p1_15["mean"] > 0 and p1_15["excl_zero"] and
         p1_1h["mean"] > 0 and p1_1h["excl_zero"])
crit2 = (m1_15["mean"] < 0 and m1_15["excl_zero"] and
         m1_1h["mean"] < 0 and m1_1h["excl_zero"])
crit3 = (m1_1h["mean"] < z0_1h["mean"] < p1_1h["mean"])

def half_sign_preserved(half_stats, state, horizon):
    if state not in half_stats:
        return False
    v = half_stats[state][horizon]
    if state == 1:
        return v > 0
    if state == -1:
        return v < 0
    return True

crit4 = all([
    half_sign_preserved(a_stats, 1, "demean_15m"),
    half_sign_preserved(a_stats, 1, "demean_1h"),
    half_sign_preserved(a_stats, -1, "demean_15m"),
    half_sign_preserved(a_stats, -1, "demean_1h"),
    half_sign_preserved(b_stats, 1, "demean_15m"),
    half_sign_preserved(b_stats, 1, "demean_1h"),
    half_sign_preserved(b_stats, -1, "demean_15m"),
    half_sign_preserved(b_stats, -1, "demean_1h"),
])

print(f"Criterion 1 (+1 CI excl zero on both horizons, positive): {'PASS' if crit1 else 'FAIL'}")
print(f"Criterion 2 (-1 CI excl zero on both horizons, negative): {'PASS' if crit2 else 'FAIL'}")
print(f"Criterion 3 (0 bucket 1h strictly between agreement buckets): {'PASS' if crit3 else 'FAIL'}")
print(f"Criterion 4 (signs preserved across both halves, both horizons): {'PASS' if crit4 else 'FAIL'}")
print()
overall = "PASS" if all([crit1, crit2, crit3, crit4]) else "FAIL"
print(f"===== OVERALL: {overall} =====")


## Plot — per-bucket means with 97.5% HAC CI error bars


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)
for ax, h in zip(axes, ["demean_15m", "demean_1h"]):
    states = [-1, 0, 1]
    means = [stats_df.loc[(s, h), "mean"] for s in states]
    errs = [Z_97_5 * stats_df.loc[(s, h), "se"] for s in states]
    colors = ["#d62728", "#999999", "#2ca02c"]
    ax.bar([str(s) for s in states], means, yerr=errs, capsize=5, color=colors, alpha=0.8)
    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_title(f"{h} (co-primary)")
    ax.set_xlabel("Composite HTF state")
    ax.set_ylabel("De-meaned mean forward return")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x*100:+.3f}%"))

plt.suptitle("HTF alignment screen — bucket means with Bonferroni 97.5% HAC CIs")
plt.tight_layout()
plt.show()
